# Paleomagnetism

The `apsg.paleomag` module reads demagnetization data of paleomagnetic specimens, fits their components by principal component analysis and provides the statistics used to interpret them: site means, combined means of directions and great circles, fold and reversal tests, virtual geomagnetic poles and inclination-only data. It is not imported by `from apsg import *`, so we import it separately.

In [ ]:
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from apsg import *
from apsg.paleomag import *

## The Teel basalts
The examples use a subset of the data of Kilian et al. (2016) from the Late Ordovician Teel Formation of the Zavkhan terrane in Mongolia, a succession of basalt flows with a magnetite remanence of probably primary origin and a hematite overprint. Every site is one flow, sampled by oriented drill cores and demagnetized thermally.

> Kilian, T.M., Swanson-Hysell, N.L., Bold, U., Crowley, J. and Macdonald, F.A. (2016). Paleomagnetism of the Teel basalts from the Zavkhan terrane: Implications for Paleozoic paleogeography in Mongolia and the growth of continental crust. *Lithosphere* 8(6), 699-715, [doi:10.1130/L552.1](https://doi.org/10.1130/L552.1).

The data are published in the [MagIC database](http://earthref.org/MAGIC/doi/10.1130/L552.1) (contribution 18692), in the GSA Data Repository (item 2016284) and in the [GitHub repository](https://github.com/Swanson-Hysell-Group/2016_Teel_Basalts) of the study. The folder `teel` next to this notebook holds 67 specimens of 8 sites converted to PMD files, the authors' component fits and the authors' site means (see its `README.md` for the conversion and its checks).

In [ ]:
sites = pd.read_csv('teel/teel_sites.csv', index_col='site')
sites[['bed_dip_direction', 'bed_dip', 'n', 'dec_geo', 'inc_geo', 'dec_tilt', 'inc_tilt', 'k_tilt']]

The table holds the bedding of every flow and the mean direction of the magnetite component calculated by the authors in geographic and in tilt-corrected coordinates. We will compare it with our own results.

## Specimens
A `Specimen` holds the demagnetization steps of one specimen with its core orientation and bedding. `Specimen.from_file` reads PMD, Remasoft RS3 and SQUID files, which have also dedicated `from_pmd`, `from_rs3` and `from_squid` constructors.

In [ ]:
spec = Specimen.from_pmd('teel/Z31.1a.pmd')
spec

The steps are available as arrays or as a table with the directions in all three coordinate systems: specimen (`sp`), geographic (`ge`) and tilt corrected (`tc`).

In [ ]:
spec.to_dataframe().head(8).round(1)

The magnetization is steep and upward in geographic coordinates after the low-temperature steps. The decay plot shows the intensity against the treatment level, and the Zijderveld diagram shows the demagnetization path in orthogonal projections in geographic and tilt-corrected coordinates.

In [ ]:
fig, (a, b, c) = plt.subplots(1, 3, figsize=(15, 5))
demag_plot(spec, ax=a)
zijderveld_plot(spec, coords='geo', ax=b)
zijderveld_plot(spec, coords='tilt', ax=c)
plt.tight_layout()

## Components
Components are fitted to a selection of steps by principal component analysis (Kirschvink, 1980). Steps are selected by slice, position, labels or boolean mask, and `between` returns the steps in a range of treatment levels. The authors fitted the components with PmagPy and gave the temperature range of every fit.

In [ ]:
magic = pd.read_csv('teel/teel_components.csv')
magic[magic.specimen == 'Z31.1a']

The MAG component is a line fitted to the steps from 530 to 582 °C. `interpret` fits and stores the component under a name, and returns it in geographic coordinates.

In [ ]:
fit = magic[(magic.specimen == 'Z31.1a') & (magic.component == 'MAG')].iloc[0]
steps = spec.between(fit.step_min, fit.step_max)
mag = spec.interpret('MAG', steps.labels)
mag

`Component` holds the direction `vector`, the maximum angular deviation `mad` and the deviation angle `dang` (the angle between the line and the center of mass of the data). They agree with the fit of the authors.

In [ ]:
print(f'apsg:    dec {mag.vector.geo[0]:.1f} inc {mag.vector.geo[1]:.1f} MAD {mag.mad:.1f} DANG {mag.dang:.1f}')
print(f'authors: dec {fit.dec_geo:.1f} inc {fit.inc_geo:.1f} MAD {fit.mad:.1f} DANG {fit.dang:.1f}')

Components are stored in specimen coordinates and rotated on demand, so the same component is available in geographic or tilt-corrected coordinates. The stereonet shows the demagnetization path of the specimen with the stored component (filled symbols are in the lower, open symbols in the upper hemisphere).

In [ ]:
spec.component('MAG', coords='tilt')

In [ ]:
stereo_plot(spec, coords='geo', show=False).show()

An anchored fit is forced through the origin, which is appropriate for a component that decays to the origin. A `kind="plane"` fit returns a great circle, described by its pole (see below).

In [ ]:
spec.pca(steps.labels, anchor=True)

## Site means
A `Sample` collects specimens, for example the specimens of one site. The function below reads all specimens of a site, fits the MAG component with the step ranges of the authors and sets the location. The mean of the authors for the flow Z51 (5 specimens) is reproduced only without the fits of the specimens Z51.1a and Z51.2a, which are based on lower temperature steps and give shallower directions, so these are left out here (`skip`).

In [ ]:
def read_site(site, skip=()):
    smp = Sample.from_files(sorted(glob(f'teel/{site}.*.pmd')), name=site)
    for s in smp:
        s.latitude, s.longitude = sites.latitude[site], sites.longitude[site]
        fit = magic[(magic.specimen == s.name) & (magic.component == 'MAG')]
        if len(fit) and s.name not in skip:
            s.interpret('MAG', s.between(fit.step_min.iloc[0], fit.step_max.iloc[0]).labels)
    return smp


skip = ['Z51.1a', 'Z51.2a']
samples = {site: read_site(site, skip) for site in sites.index}
samples['Z31']

`mean` gives the Fisher statistics of the components in any coordinate system (`"specimen"`, `"geo"` or `"tilt"`): mean direction `mu`, precision `k` and the radius `alpha` of the 95% confidence cone.

In [ ]:
samples['Z31'].mean('MAG', coords='geo')

In [ ]:
samples['Z31'].mean('MAG', coords='tilt')

The precision does not change with the tilt correction of a single flow, as it is a rotation of all specimens. We compare the means of all sites with the means calculated by the authors.

In [ ]:
rows = []
for site, smp in samples.items():
    for coords, name in (('geo', 'geo'), ('tilt', 'tilt')):
        m = smp.mean('MAG', coords)
        a = sites.loc[site]
        rows.append({
            'site': site, 'coords': name, 'n': m['n_lines'],
            'dec': m['mu'].geo[0], 'inc': m['mu'].geo[1], 'alpha95': m['alpha'], 'k': m['k'],
            'dec_authors': a[f'dec_{name}'], 'inc_authors': a[f'inc_{name}'],
            'alpha95_authors': a[f'alpha95_{name}'], 'k_authors': a[f'k_{name}'],
        })
means = pd.DataFrame(rows)
means.round(1)

All site means reproduce those of the authors. For the mean of the flow Z51 to agree, the two specimens had to be left out: with all seven fits the precision is much lower.

In [ ]:
print('all 7 specimens:', round(read_site('Z51').mean('MAG', 'geo')['k'], 1))
print('5 specimens:    ', round(samples['Z51'].mean('MAG', 'geo')['k'], 1))

The directions of all specimens in geographic and in tilt-corrected coordinates, with the specimens of a flow sharing the same bedding. Filled symbols are in the lower, open symbols in the upper hemisphere.

In [ ]:
everything = Sample([s for smp in samples.values() for s in smp], name='Teel')
s = StereoNet(title='Geographic')
s.vector(everything.directions('MAG', 'geo'), color='C0', alpha=0.6)
s.show()

In [ ]:
s = StereoNet(title='Tilt corrected')
s.vector(everything.directions('MAG', 'tilt'), color='C1', alpha=0.6)
s.show()

The mean of all specimens in the two coordinate systems. The flows have different bedding, so the precision of the whole set changes with the tilt correction, and it increases. Note that the specimens of a flow are not independent samples of the flows, so the confidence cones of the whole set are too small.

In [ ]:
everything.mean('MAG', 'geo')

In [ ]:
everything.mean('MAG', 'tilt')

## Great circles and combined means
When a component cannot be isolated, the demagnetization path of a specimen follows a great circle toward the sought direction, which gives a constraint instead of a point. A `kind="plane"` fit returns the great circle, described by its pole. In the flow Z36 we replace the line fits of three specimens, whose paths follow well defined circles (fitted to the steps from 200 to 545 °C), by the circles. The circles pass within a few degrees of the line fits.

In [ ]:
z36 = read_site('Z36')
for name in ['Z36.1a', 'Z36.5a', 'Z36.6a']:
    s = z36[name]
    line = s.component('MAG', 'geo')
    circle = s.interpret('MAG', s.between(200, 545).labels, kind='plane')
    dist = abs(90 - circle.vector.angle(line.vector))
    print(f'{name}: MAD of the circle {circle.mad:.1f}, line {dist:.1f} degrees from the circle')

`Sample.mean` combines the directions and the great circles by the method of McFadden and McElhinny (1988): each circle contributes its point closest to the mean, iterated to convergence, and the precision and the confidence cone account for the weaker constraint of the circles. We compare it with the Fisher mean of the line fits of all specimens.

In [ ]:
combined = z36.mean('MAG', 'geo')
lines_only = read_site('Z36').mean('MAG', 'geo')
for label, m in (('lines only', lines_only), ('lines and circles', combined)):
    dec, inc = m['mu'].geo
    print(f"{label:18s} dec {dec:.1f} inc {inc:.1f} alpha95 {m['alpha']:.1f} ({m['n_lines']} lines, {m['n_planes']} circles)")

The underlying functions work on arrays of directions and poles. `kirschvink_mean` is the least-squares method of Kirschvink (1980), which has no confidence region, and `bailey_halls_mean` is the maximum likelihood solution of Bailey and Halls (1984), which gives a confidence ellipse. Great circles alone give a biased mean when they come from two populations (Schmidt, 1985), so direct observations of the sought direction should be included.

In [ ]:
comps = [s.component('MAG', 'geo') for s in z36]
lines = [c.vector for c in comps if c.kind == 'line']
poles = [c.vector for c in comps if c.kind == 'plane']
kirschvink_mean(lines, poles)['mu'].geo

In [ ]:
bh = bailey_halls_mean(lines, poles)
print(bh['mu'].geo, f"k {bh['k']:.0f}, ellipse semi-axes {bh['semi_axes'][0]:.1f} x {bh['semi_axes'][1]:.1f}")

## Fold tests
A magnetization older than the folding has the same direction in all flows after the tilt correction. The tilt of the flows in our subset varies only between 28 and 58 degrees, so a fold test has limited power, as the authors also discuss. The bootstrap fold test of Tauxe and Watson (1994) finds the percentage of untilting at which the largest eigenvalue of the orientation tensor of the directions is maximal (the directions are the most clustered), for many resamples of the data. A confidence interval of the maxima that includes 100% supports a pre-folding magnetization.

In [ ]:
ft = everything.fold_test('MAG', n=1000, rng=0)
print(f"best untilting {ft['best']:.0f} %, 95 % interval {ft['ci'][0]:.0f} to {ft['ci'][1]:.0f} %")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ft['fracs'], ft['tau1'])
ax.axvspan(*ft['ci'], alpha=0.2, label='95 % interval of the maxima')
ax.set_xlabel('Untilting [%]')
ax.set_ylabel(r'$\tau_1$')
ax.legend()

The authors, who used the 23 flows of their pole, obtained 49 to 95 % untilting at the 95 % confidence level (41 to 101 % at 99 %). Our interval from eight flows is shifted toward complete untilting; with so few flows and so little variation in the tilt, the difference is not meaningful, and the authors also note the limited significance of the tilt test.

The test of McFadden (1990) is designed for flows with their own bedding, each one on its own limb of a fold. For a chosen percentage of untilting it calculates the correlation `xi` between the deviations of the directions from their mean and the displacement of the mean by a small tilt about the strike of each bed. Directions that scatter because they are tilted differently correlate with the tilt, so a large `|xi|` shows that they are not in their primary position. The statistic is compared with a critical value of the null distribution (Lord, 1948), and the percentage of untilting with the smallest `|xi|` is the best estimate. A magnetization acquired before the folding is significant in situ but not when unfolded. We use the mean directions of the eight flows.

In [ ]:
directions = [np.asarray(smp.mean('MAG', 'geo')['mu']) for smp in samples.values()]
beddings = [np.asarray(smp[0].bedding) for smp in samples.values()]
mf = mcfadden_fold_test(directions, beddings)
print(f"xi in situ {mf['geo']:.2f}, unfolded {mf['tilt']:.2f}, critical value {mf['critical']:.2f}, N = {mf['n']}")
print(f"smallest |xi| at {mf['best']:.0f} % untilting")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(mf['fracs'], mf['xi'])
ax.axhline(mf['critical'], color='r', ls='--', lw=1)
ax.axhline(-mf['critical'], color='r', ls='--', lw=1, label='critical value')
ax.set_xlabel('Untilting [%]')
ax.set_ylabel(r'$\xi$')
ax.legend()

With eight flows and tilts between 28 and 58 degrees, neither the in situ nor the unfolded position is significant, so the test cannot distinguish them, which agrees with the authors' conclusion that the tilt test is weak. The same test can be run on the specimens of a `Sample` with `Sample.mcfadden_fold_test`, treating every specimen as a site.

## Virtual geomagnetic pole
`Sample.vgp` calculates the virtual geomagnetic pole of the mean direction with its oval of confidence `dp` and `dm` and the paleolatitude of the site. The plain function `vgp` does the same for a direction. The confidence oval is small, because it is calculated from the specimens, which are not independent samples of the flows. The authors calculated the pole as the mean of the site poles of 23 flows (pole at -36.4, 16.2 and paleolatitude -19.1 degrees), so a similar but not identical result is expected.

In [ ]:
everything.vgp('MAG', coords='tilt', latitude=sites.latitude.mean(), longitude=sites.longitude.mean())

The site mean of one flow gives its virtual pole. We compare the pole of the flow Z31 with that of the authors.

In [ ]:
m = samples['Z31'].mean('MAG', 'tilt')
dec, inc = m['mu'].geo
pole = vgp(dec, inc, sites.latitude['Z31'], sites.longitude['Z31'], m['alpha'])
print(f"apsg:    lat {pole['lat']:.1f} lon {pole['lon']:.1f} paleolatitude {pole['paleolatitude']:.1f}")
print(f"authors: lat {sites.vgp_lat['Z31']:.1f} lon {sites.vgp_lon['Z31']:.1f} paleolatitude {sites.paleolatitude['Z31']:.1f}")

## Inclination-only data
Sometimes only inclinations are known, for example for unoriented drill cores. `inclination_only_mean` gives the mean inclination of Fisher distributed directions with unknown declination, the precision and the error angle (McFadden and Reid, 1982). We pretend that the declinations of the tilt-corrected MAG directions (taken to the lower hemisphere) are lost and compare the estimate with the full directions.

In [ ]:
v = np.array([-np.asarray(c.vector) for c in
              (s.component('MAG', 'tilt') for s in everything if 'MAG' in s.components)])
inc = np.degrees(np.arcsin(v[:, 2] / np.linalg.norm(v, axis=1)))
full = vecset([vec(*x) for x in v]).fisher_statistics()
only = inclination_only_mean(inc)
print(f"full directions: inc {full['mu'].geo[1]:.1f} alpha95 {full['alpha']:.1f}")
print(f"inclination only: inc {only['inc']:.1f} alpha {only['alpha']:.1f} k {only['k']:.0f}")

## Saving specimens
`Specimen` and `Sample` have `to_json` for a lossless copy, and a specimen can be written to the PMD and RS3 formats by `to_pmd` and `to_rs3`. The API of the module is listed in the [reference](../api/paleomag.md).